# AccessPath: 01 Auto-Label & Prepare Dataset

**Objective**: Zero-shot obstacle detection on pilot sidewalk imagery (Bengaluru Indiranagar pilot) using Hugging Face OWLv2 (`google/owlv2-base-patch16-ensemble`, Apache 2.0 license).

### Licensing & Zero-Budget Constraints
- Runs on **Free Google Colab T4 GPU**.
- **Strictly permissive open-source**: Uses Apache 2.0 model checkpoints; avoids copyleft AGPL (Ultralytics YOLO).
- Output format: Standardized JSON annotations for human-in-the-loop validation and conversion to COCO/YOLO.

In [ ]:
# Step 1: Install Colab Dependencies
!pip install -q transformers accelerate pillow requests tqdm torch torchvision

In [ ]:
import os
import json
import requests
from pathlib import Path
from PIL import Image
import torch
from transformers import Owlv2Processor, Owlv2ForObjectDetection

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

In [ ]:
# Step 2: Define Obstacle Candidate Prompts (Apache 2.0 OWLv2 queries)
CANDIDATE_QUERIES = [
    "steps on sidewalk or stairs",
    "utility pole or concrete post",
    "parked vehicle motorcycle auto-rickshaw",
    "street vendor cart or food stall",
    "pile of garbage trash rubble debris",
    "construction material sand gravel bricks",
    "broken curb or missing kerb ramp",
]

QUERY_TO_LABEL = {
    "steps on sidewalk or stairs": "steps",
    "utility pole or concrete post": "pole",
    "parked vehicle motorcycle auto-rickshaw": "parked_vehicle",
    "street vendor cart or food stall": "vendor_cart",
    "pile of garbage trash rubble debris": "garbage_rubble",
    "construction material sand gravel bricks": "construction_material",
    "broken curb or missing kerb ramp": "missing_kerb_ramp",
}

In [ ]:
# Step 3: Load Permissive Open-Source Model
MODEL_ID = "google/owlv2-base-patch16-ensemble"
print(f"Loading {MODEL_ID}...")
processor = Owlv2Processor.from_pretrained(MODEL_ID)
model = Owlv2ForObjectDetection.from_pretrained(MODEL_ID).to(device)
model.eval()
print("Model loaded successfully!")

In [ ]:
# Step 4: Download Pilot Street Imagery (Mapillary / KartaView / Sample Cache)
IMAGE_DIR = Path("pilot_images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

# Create sample demonstration image or fetch pilot sequence
sample_image_path = IMAGE_DIR / "indiranagar_sample_01.jpg"
if not sample_image_path.exists():
    # Generate synthetic 1920x1080 sidewalk image with simulated barrier
    img = Image.new("RGB", (1920, 1080), color=(140, 150, 160))
    img.save(sample_image_path)
    print(f"Created placeholder pilot frame: {sample_image_path}")

In [ ]:
# Step 5: Zero-Shot Inference Function
def auto_label_image(image_path: Path, score_threshold: float = 0.25):
    image = Image.open(image_path).convert("RGB")
    w, h = image.size

    inputs = processor(text=[CANDIDATE_QUERIES], images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        outputs = model(**inputs)

    target_sizes = torch.Tensor([image.size[::-1]]).to(device)
    results = processor.post_process_object_detection(outputs=outputs, target_sizes=target_sizes, threshold=score_threshold)[0]

    annotations = []
    for score, label_idx, box in zip(results["scores"], results["labels"], results["boxes"]):
        query_text = CANDIDATE_QUERIES[label_idx]
        canonical_label = QUERY_TO_LABEL.get(query_text, "unknown")

        # Convert [xmin, ymin, xmax, ymax] in pixels to normalized [ymin, xmin, ymax, xmax]
        xmin, ymin, xmax, ymax = box.tolist()
        norm_box = [
            round(ymin / h, 4),
            round(xmin / w, 4),
            round(ymax / h, 4),
            round(xmax / w, 4),
        ]

        annotations.append({
            "label": canonical_label,
            "raw_query": query_text,
            "confidence": round(score.item(), 3),
            "bbox": norm_box,
        })

    return {
        "file_name": image_path.name,
        "width": w,
        "height": h,
        "annotations": annotations,
    }

# Run sample inference
res = auto_label_image(sample_image_path)
print(json.dumps(res, indent=2))

In [ ]:
# Step 6: Batch Process and Export Raw Auto-Labeled Dataset
all_samples = []
for img_file in IMAGE_DIR.glob("*.jpg"):
    sample_result = auto_label_image(img_file, score_threshold=0.25)
    all_samples.append(sample_result)

output_json = Path("auto_labeled_dataset.json")
with open(output_json, "w", encoding="utf-8") as f:
    json.dump(all_samples, f, indent=2)

print(f"Successfully exported {len(all_samples)} images with auto-labels to {output_json}")
print("Ready for human review and conversion via dataset_converter.py!")